# COMP0178 - Lesson 2: Database Design

- Explained notes covering **all 71 PDF pages** of *2. Database Design*, Week 2.
- Source: [Lecture 2 PDF](../Lectures/Lecture_2.pdf). References use **PDF page order**, because printed slide numbers have gaps.
- Three main skills: **draw a conceptual model**, **translate it into tables**, and **check/repair the resulting design with normalization**.
- Original lecture diagrams appear next to their explanations; click an image to open it full size.
- Worked examples and practice are labelled supplementary. Qualifications identify where the slides give a simplified rule.
- Build on [Lesson 1a: database foundations](lesson1a.ipynb) and [Lesson 1b: SQL](lesson1b.ipynb).


## 1. Requirements and the design process (pages 1-6)

- **Requirements gathering:** find out what data is needed, who uses it, and what operations the system must support. Write down business rules as well as example records.
- **Conceptual design:** describe entities, attributes, relationships and constraints without committing to a DBMS or table layout. An ER diagram is a model of the requirements.
- **Logical design:** translate that model into a relational schema: tables, columns, primary keys and foreign keys. Normalize the design to remove problematic redundancy.
- Logical design is independent of a particular DBMS. Physical choices such as indexes and storage belong to a later stage.
- **Top-down:** start from an ER diagram and translate it. **Bottom-up:** start from a large table and decompose it using dependencies.
- The coursework brief supplies much of the requirements gathering; ambiguous details still need explicit assumptions.

**Supplementary example:** “Clients can view many properties; properties can be viewed by many clients; each viewing has a date and comment.” This identifies two entity types and a relationship carrying its own attributes.


## 2. Entities, relationships and UML notation (pages 6-12)

- **Entity type:** a category of things about which we store data, such as `Staff` or `Client`. An **entity instance** is one particular staff member or client.
- **Relationship:** an association, such as a Branch having Staff. It can connect more than two entity types.
- **Attribute:** a property of an entity or relationship, such as salary or the date of a viewing.
- In the lecture's UML-style notation, entities are rectangles with **UpperCamelCase** names. A relationship is a labelled line; a small arrow beside the name gives its reading direction, not a foreign-key direction.
- **Degree:** number of participating entity roles. Binary = two, ternary = three, quaternary = four. A recursive relationship can involve one entity type in multiple roles.
- Higher-degree relationships use a named diamond connected to all participants. `Registers(Staff, Client, Branch)` stores who registered whom and where; splitting it into unrelated binary links can lose this combination.
- **Role names** explain participation, particularly self-relationships: a Staff member can be a *supervisor* of another Staff member who is the *supervisee*.
- Different relationships can connect the same entity types. Give each a meaningful name and roles rather than merging them.


**Original lecture visual — PDF page 9**

[![Higher-degree relationships](../assets/slide-images/Lecture_2-page-009.png)](../assets/slide-images/Lecture_2-page-009.png)

The ternary relationship records staff, client and branch together.

Source: [Lecture_2.pdf, PDF page 9](../Lectures/Lecture_2.pdf#page=9).


**Original lecture visual — PDF page 11**

[![Role names](../assets/slide-images/Lecture_2-page-011.png)](../assets/slide-images/Lecture_2-page-011.png)

Read the role at each endpoint to distinguish participants in a recursive relationship.

Source: [Lecture_2.pdf, PDF page 11](../Lectures/Lecture_2.pdf#page=11).


## 3. Keys and kinds of attributes (pages 13-15)

| Term | Meaning | Example |
| --- | --- | --- |
| Superkey | Any attribute set that uniquely identifies an instance | `{staffNo, name}` when staffNo alone is unique |
| Candidate key | A **minimal** superkey: no attribute can be removed | `{staffNo}`; possibly another guaranteed unique identifier |
| Primary key | The candidate key selected as the main identifier | `staffNo` |
| Composite key | A key containing several attributes | `{clientNo, propertyNo, viewDate}` if this identifies a viewing |
| Composite attribute | One conceptual attribute with meaningful parts | Address → street, city, postcode |
| Multi-valued attribute | An instance may have multiple values | Branch phone numbers |
| Derived attribute | Computable from stored data | monthlySal = salary / 12 |

- A composite **key** identifies something; a composite **attribute** describes something. They are different concepts.
- A key must be unique under the requirements, not just in today's sample rows. A person's name is usually unsuitable.
- Lecture notation: `{PK}` marks a primary key, `{CK}` a candidate key, `{PPK}` parts of a composite primary key; `/` prefixes a derived attribute; `[1..3]` means one to three values.
- Composite attribute parts are indented below their heading. Flatten them to separate columns during translation.
- **Relationship attributes** describe the association: advertisement date and cost belong to the Newspaper–Property advertisement, rather than Newspaper alone or Property alone.
- If repeated advertisements or viewings are allowed, their dates or an event ID may be needed in the relationship key.


**Original lecture visual — PDF page 14**

[![Attributes](../assets/slide-images/Lecture_2-page-014.png)](../assets/slide-images/Lecture_2-page-014.png)

Locate the primary key, composite address, multi-valued phone numbers and derived monthly salary.

Source: [Lecture_2.pdf, PDF page 14](../Lectures/Lecture_2.pdf#page=14).


**Original lecture visual — PDF page 15**

[![Relationship attributes](../assets/slide-images/Lecture_2-page-015.png)](../assets/slide-images/Lecture_2-page-015.png)

The attribute box belongs to the relationship rather than either entity alone.

Source: [Lecture_2.pdf, PDF page 15](../Lectures/Lecture_2.pdf#page=15).


## 4. Read multiplicities from the opposite end (pages 16-18)

**First fix one instance at one end; then read the range written near the other end.**

```text
Staff   1..1 -------- Manages -------- 0..*   Property
```

- Fix **one Staff**: the range near Property is `0..*`, so that staff member manages zero or more properties.
- Fix **one Property**: the range near Staff is `1..1`, so that property has exactly one manager.
- **Cardinality** is the maximum; **participation** is the minimum. `0` means optional participation; a minimum of `1` or more means mandatory participation.
- `1..1` may be written `1`; `0..*` may be written `*`. The star means **no specified upper bound**, not that infinitely many records actually exist.

| Range | Interpretation |
| --- | --- |
| `0..1` | None or one |
| `1..1` | Exactly one |
| `0..*` | None, one, or many |
| `1..*` | At least one |
| `1..3` | One, two, or three |

- **1:1**, **1:many**, and **many:many** describe upper bounds. They do not specify whether participation is optional.
- Common mistake: reading `1` near Staff as “a staff member manages one property.” It actually describes staff **per property**.


**Original lecture visual — PDF page 17**

[![Multiplicity](../assets/slide-images/Lecture_2-page-017.png)](../assets/slide-images/Lecture_2-page-017.png)

Fix one Staff and read the Property endpoint; then fix one Property and read the Staff endpoint.

Source: [Lecture_2.pdf, PDF page 17](../Lectures/Lecture_2.pdf#page=17).


## 5. Semantic nets and higher-order multiplicity (pages 19-25)

- An **ER diagram** describes entity types and rules. A **semantic net** draws particular records and their connections to illustrate the rules.
- In a 1:many example, one branch connects to several staff records, while each staff record connects to its branch.
- In the lecture's 1:1 manager example, each branch has one manager and a staff member manages at most one branch; not every staff member must be a manager.
- In a many:many advertisement example, both a newspaper and a property may have multiple associations. Read the lower bounds too.
- For a **ternary** relationship, hold **the other two participants fixed** to interpret a bound at an endpoint.
- For `Registers(Staff, Branch, Client)` with `1` near Staff, `1` near Branch and `*` near Client:
  - Fix Staff + Branch: many clients may be registered.
  - Fix Staff + Client: at most one branch is associated with that pair.
  - Fix Branch + Client: at most one staff member is associated with that pair.
- These rules alone do **not** imply that a client has only one registration globally. Do not replace a ternary relationship with binary ones or choose its key without checking the full requirements.


**Original lecture visual — PDF page 19**

[![Semantic nets](../assets/slide-images/Lecture_2-page-019.png)](../assets/slide-images/Lecture_2-page-019.png)

These are individual records illustrating a one-to-many relationship, rather than entity types.

Source: [Lecture_2.pdf, PDF page 19](../Lectures/Lecture_2.pdf#page=19).


**Original lecture visual — PDF page 21**

[![One-to-one](../assets/slide-images/Lecture_2-page-021.png)](../assets/slide-images/Lecture_2-page-021.png)

Read both minimum and maximum values; one-to-one does not by itself require participation.

Source: [Lecture_2.pdf, PDF page 21](../Lectures/Lecture_2.pdf#page=21).


**Original lecture visual — PDF page 23**

[![Many-to-many](../assets/slide-images/Lecture_2-page-023.png)](../assets/slide-images/Lecture_2-page-023.png)

Both sides can have multiple related instances.

Source: [Lecture_2.pdf, PDF page 23](../Lectures/Lecture_2.pdf#page=23).


**Original lecture visual — PDF page 24**

[![Ternary multiplicity](../assets/slide-images/Lecture_2-page-024.png)](../assets/slide-images/Lecture_2-page-024.png)

Hold the other two participants fixed when interpreting an endpoint bound.

Source: [Lecture_2.pdf, PDF page 24](../Lectures/Lecture_2.pdf#page=24).


**Original lecture visual — PDF page 25**

[![Ternary semantic net](../assets/slide-images/Lecture_2-page-025.png)](../assets/slide-images/Lecture_2-page-025.png)

The left side contains fixed Staff/Branch pairs; the right side contains clients.

Source: [Lecture_2.pdf, PDF page 25](../Lectures/Lecture_2.pdf#page=25).


### Practice A — Explain multiplicity before revealing the answer

A Customer places Orders. A customer can have zero or more orders; every order belongs to exactly one customer.

1. What range belongs near Customer? Near Order?
2. Is this 1:1, 1:many or many:many?
3. Which table will contain the foreign key?


In [ ]:
# Write your explanation, then run this cell.
answer_a = ""
print(answer_a if answer_a.strip() else "Write your answer in answer_a.")


<details><summary>Answer and reasoning</summary>

- Near Customer: **1**; near Order: **0..***. Read each from a fixed instance at the other end.
- It is **1:many**. Customer participation is optional; Order participation is mandatory.
- Order contains a **NOT NULL customerID foreign key**. It is not UNIQUE, because several orders may belong to the same customer.

</details>


## 6. Enhanced ER: specialization and generalization (pages 26-30)

- **EER** extends ER with superclasses, subclasses and inheritance.
- **Specialization:** split a general type into more specific types; Staff → Manager, Salesperson, Secretary.
- **Generalization:** recognize common properties in existing types and introduce a superclass.
- Subclasses inherit their superclass's attributes and relationships and may add their own. A Manager is still a Staff instance.
- **Participation:** mandatory/total means every superclass instance belongs to at least one subclass; optional/partial allows superclass instances with no subclass.
- **Disjointness:** disjoint (`or`) allows at most one subclass; overlapping/nondisjoint (`and`) allows multiple subclasses.

| Constraints | Allowed subclass memberships per superclass instance |
| --- | --- |
| Mandatory + disjoint | Exactly one |
| Optional + disjoint | Zero or one |
| Mandatory + overlapping | One or more |
| Optional + overlapping | Zero or more |

- The inheritance arrow points **from subclass to superclass**. Labels in braces specify the constraints; the arrow alone does not specify them.


**Original lecture visual — PDF page 28**

[![Specialization](../assets/slide-images/Lecture_2-page-028.png)](../assets/slide-images/Lecture_2-page-028.png)

Subclasses inherit the shared Staff attributes and add their own.

Source: [Lecture_2.pdf, PDF page 28](../Lectures/Lecture_2.pdf#page=28).


**Original lecture visual — PDF page 30**

[![Inheritance constraints](../assets/slide-images/Lecture_2-page-030.png)](../assets/slide-images/Lecture_2-page-030.png)

Read total/partial participation separately from disjoint/overlapping membership.

Source: [Lecture_2.pdf, PDF page 30](../Lectures/Lecture_2.pdf#page=30).


## 7. Translate entities and weak entities into tables (pages 31-36)

- Each **strong entity** becomes a table containing its simple stored attributes, with its selected primary key.
- Flatten a composite address into street, city and postcode. Usually calculate derived attributes rather than storing duplicate values that can disagree.
- A **weak entity** depends on an owner for existence; its identity may include the owner's key and a partial identifier. In strict ER terminology, identifying dependence matters as well as existence dependence.
- Lecture example: `Client(clientNo, fName, lName, telNo)` and `Preference(clientNo, prefType, maxRent)`.
- If there is exactly one Preference record per client, clientNo can be both **PK and FK**. If several preferences are allowed, another discriminator is needed in its key.
- Do not assume that every child table is weak: an ordinary Order can have its own independent orderID even though it refers to a Customer.


## 8. Translate one-to-many and one-to-one relationships (pages 37-41)

**1:many:** put the one-side key into the many-side table as a foreign key. Put the relationship attributes there too when each child has one relationship instance.

```text
Staff(staffNo PK, fName, lName, ...)
Client(clientNo PK, fName, lName, ..., staffNo FK → Staff)
```

- Make the FK NOT NULL when that child must participate. A foreign key alone checks a referenced row exists; it does not require a non-NULL value.
- A parent FK in the child does not automatically enforce a parent's minimum of one child.

| 1:1 participation | Lecture recommendation | Constraint to remember |
| --- | --- | --- |
| Mandatory on both sides | Combine into one table where sensible | Choose a PK; preserve other candidate keys |
| Mandatory on one side | Mandatory participant holds optional participant's key | FK is NOT NULL and UNIQUE, or a shared PK/FK |
| Optional on both sides | Choose a side to hold the other key | Nullable UNIQUE FK, or a separate link table |

- **UNIQUE** is necessary to stop multiple rows pointing to the same partner in a 1:1 association.
- Keeping two tables for mandatory 1:1 can be justified by lifecycle or access requirements; mutual total participation needs additional enforcement beyond a one-way FK.
- The lecture's “FK works as PK” is the shared-key option. A child may instead keep its own PK and use a separate UNIQUE FK.


## 9. Recursive, many-to-many and higher-order relationships (pages 42-44)

- **Recursive relationship:** a table refers to another row in itself. Use distinct role names, for example `Staff(staffNo PK, supervisorNo FK → Staff)`.
- For recursive **1:1**, a self-FK needs UNIQUE to cap incoming links; NULL handles optional participation. A separate relation such as `Partnership(personA FK, personB FK)` can make the two roles explicit.
- The slide's recursive advice has overlapping cases (“one or both sides”); choose from the actual minimum/maximum constraints, not wording alone. Symmetry, forbidding self-links and ensuring one partner across both roles need extra constraints.
- **Many:many:** create a junction/relationship table holding the participants' foreign keys and the relationship attributes.

```text
Viewing(clientNo FK, propertyNo FK, viewDate, comment)
PK = (clientNo, propertyNo, viewDate)  -- if repeated visits on different dates are allowed
```

- With at most one viewing per client/property pair, `(clientNo, propertyNo)` suffices instead. State the rule.
- **Ternary/higher-degree:** create one relationship table with all participant keys and relationship attributes, for example `Registration(clientNo, branchNo, staffNo, dateJoined)`.
- Derive its PK from the functional dependencies. Without tighter constraints, the participant-key combination is the starting point; repeated events can require another discriminator.
- The lecture suggests clientNo for its registration example. That requires the **additional rule that each client has only one registration overall**; the endpoint multiplicities alone do not prove it.


## 10. Translate inheritance and multi-valued attributes (pages 45-49)

The lecture offers **recommendations**, not one mandatory implementation for every design:

| EER constraints | Lecture's suggested layout |
| --- | --- |
| Mandatory + overlapping (`and`) | One combined table; membership flags and nullable subtype columns |
| Mandatory + disjoint (`or`) | One table for each superclass–subclass combination, repeating common attributes |
| Optional + overlapping | Superclass table plus a combined subtype-details table with flags |
| Optional + disjoint | Superclass table plus one table per subclass, linked by PK/FK |

- Another common layout uses a superclass table and separate PK/FK-linked subclass tables for overlapping membership too; this is supplementary, rather than the slide's chosen layout.
- Decide how membership constraints will actually be enforced. Foreign keys alone do not ensure total or disjoint membership.
- **Multi-valued attributes:** create a separate table containing the owner key and one value per row.

```text
Branch(branchNo PK, street, city, postcode)
BranchPhone(branchNo FK → Branch, telNo, PK(branchNo, telNo))
```

- Owner key alone cannot identify multiple phone numbers. The composite key prevents repeating a phone number for the same branch.
- A range such as `[1..3]` needs more than this PK/FK to enforce at least one and at most three rows.
- After translation, verify keys, relationship constraints and normal forms. ER translation does **not automatically prove 3NF** for arbitrary dependencies.


**Original lecture visual — PDF page 48**

[![Translation summary](../assets/slide-images/Lecture_2-page-048.png)](../assets/slide-images/Lecture_2-page-048.png)

Use this as a checklist, then check keys and constraints against the requirements.

Source: [Lecture_2.pdf, PDF page 48](../Lectures/Lecture_2.pdf#page=48).


## 11. Why normalize? Anomalies and decomposition (pages 50-55)

- **Normalization** restructures tables to reduce redundant facts that cause inconsistent or unintended changes.
- **Update anomaly:** a property's address is repeated across leases; changing only one copy leaves conflicting addresses.
- **Insertion anomaly:** a new property cannot be recorded in a lease-only table until it has a lease.
- **Deletion anomaly:** deleting a property's last lease also deletes the only record of its address.
- **1NF:** one value at each row/column intersection; no repeating groups or lists of phone numbers in one cell.
- Flatten repeating groups into rows or move them to their own table. Choose the new key carefully: the owner's PK is generally **only part** of the key when several values are possible.
- **Lossless join:** joining the decomposed tables reconstructs exactly the original legal relation, with no lost or invented combinations.
- **Dependency preservation:** the original FDs can be enforced by dependencies on the component tables, without joining them to check the original constraint.
- These are **separate requirements**. Normalizing carelessly can produce a lossy decomposition; BCNF decompositions can lose dependency preservation. A suitable 3NF synthesis can preserve dependencies and be lossless.
- Normal-form hierarchy: **BCNF ⇒ 3NF ⇒ 2NF ⇒ 1NF**. A higher form includes the requirements of lower forms.


## 12. Functional dependencies and attribute closure (pages 56-59; supplement)

- **X → Y** means that any two legal rows agreeing on X must agree on Y. X is the **determinant**.
- It follows from the business rules for **all legal database states**, not an accidental pattern in today's data.
- `staffNo → name, position` is plausible. `position → staffNo` is generally false: many staff can hold the same position.
- The PDF's text extraction can lose “does not determine” symbols. Do not read its negative examples as asserted FDs.
- **Full dependency:** no proper subset of X still determines Y. **Partial dependency:** a proper subset does.
- A dependency is **trivial** if Y is a subset of X; these do not cause normal-form violations.
- **Supplementary method: closure X⁺.** Start with X. Repeatedly add the RHS of any FD whose LHS is already included. Stop when nothing changes.
- X is a superkey when X⁺ contains every attribute. It is a candidate key only if it is also minimal.

**Example:** for `Lease(clientNo, propNo, cName, address, leaseStart)` assume:

```text
clientNo → cName
propNo → address
(clientNo, propNo) → leaseStart
```

- `{clientNo, propNo}⁺` contains all five attributes.
- Removing either component loses facts, so the pair is a candidate key.
- cName and address depend on only part of this composite key; leaseStart needs both under these assumptions.
- Real lease systems may allow repeated leases for the same pair. That would require a different key, such as an event ID or a date component.


### Practice B — Check an FD on sample rows (supplementary runnable tool)

Predict whether `staffNo → position` and `position → staffNo` hold **in this sample**. Passing a sample check does not prove an FD is a business rule.


In [ ]:
sample_staff = [
    {"staffNo": "S1", "position": "Manager"},
    {"staffNo": "S2", "position": "Assistant"},
    {"staffNo": "S3", "position": "Assistant"},
]

def fd_holds(rows, left, right):
    seen = {}
    for row in rows:
        determinant = tuple(row[name] for name in left)
        dependent = tuple(row[name] for name in right)
        if determinant in seen and seen[determinant] != dependent:
            return False
        seen[determinant] = dependent
    return True

print("staffNo → position:", fd_holds(sample_staff, ["staffNo"], ["position"]))
print("position → staffNo:", fd_holds(sample_staff, ["position"], ["staffNo"]))
# Expected: True, False.


## 13. Remove partial dependencies: 1NF → 2NF (pages 56-61)

- **2NF:** the relation is in 1NF, and every **non-prime attribute** depends fully on **every candidate key**. A prime attribute belongs to at least one candidate key.
- The lecture uses “non-primary-key attribute” and the chosen PK as a simplified case. With alternative candidate keys, check all of them.
- With only single-attribute candidate keys, there is no nonempty proper key subset to cause the usual partial-dependency violation.
- For the Lease example above, assuming no additional FDs, move client and property facts into their own tables:

```text
Client(clientNo PK, cName)
Property(propNo PK, address)
Lease(clientNo FK, propNo FK, leaseStart, PK(clientNo, propNo))
```

- A client name is stored once per client, not once per lease; a property address is stored once per property.
- Keep the determinant columns in Lease as foreign keys so the original rows can be reconstructed.
- Removing partial dependencies does not remove every possible transitive dependency; check 3NF next.


**Original lecture visual — PDF page 60**

[![2NF decomposition](../assets/slide-images/Lecture_2-page-060.png)](../assets/slide-images/Lecture_2-page-060.png)

Move client and property facts into their own tables; retain the relationship facts in Lease.

Source: [Lecture_2.pdf, PDF page 60](../Lectures/Lecture_2.pdf#page=60).


## 14. Remove problematic transitive dependencies: 2NF → 3NF (pages 62-65)

- A transitive chain is **A → B and B → C**, so A → C.
- In the lecture's usual example, A is the key while B is not a superkey and C is non-prime. Storing all three repeats the B–C fact.

```text
Inspection(propertyNo, inspectionDate, staffNo, staffName)
(propertyNo, inspectionDate) → staffNo
staffNo → staffName
```

Assume the pair is the only candidate key. The staff name depends on it **through staffNo**, so this relation violates 3NF.

```text
Inspection(propertyNo FK, inspectionDate, staffNo FK,
           PK(propertyNo, inspectionDate))
Staff(staffNo PK, staffName)
```

- Keep staffNo in Inspection; move staffName into Staff. Changing a staff name now changes one row.
- **Precise 3NF rule:** for every nontrivial FD X → A with a single RHS attribute, **X is a superkey OR A is prime**.
- The “no transitive dependencies” slogan is a shortcut, not the full definition. A key determining another key is not automatically a violation.


## 15. BCNF: the determinant must be a superkey (pages 66-69)

- **BCNF:** for every nontrivial FD X → Y, X must be a **superkey**. The slide says candidate key; superkey is the precise rule because nonminimal superkeys are allowed too.
- The lecture's overlapping composite-key pattern is a common way a 3NF relation fails BCNF, not a complete test for all BCNF violations.

**Lecture example:** `Teaching(pupil, subject, teacher)` with:

```text
(pupil, subject) → teacher
teacher → subject
```

- Candidate keys: `{pupil, subject}` and `{pupil, teacher}`. All three attributes are prime.
- teacher⁺ = `{teacher, subject}`, missing pupil. Therefore teacher is **not a superkey**.
- teacher → subject passes 3NF because subject is prime, but fails BCNF because teacher is not a superkey.
- Repeating a teacher's subject across pupil rows causes update anomalies.

Decompose into:

```text
Teaching(pupil, teacher, PK(pupil, teacher))
TaughtSubject(teacher PK, subject)
```

- The decomposition is lossless: the shared teacher determines TaughtSubject.
- Under the stated FDs, both components are in BCNF.
- **Dependency preservation is lost:** `(pupil, subject) → teacher` spans both tables. A pupil assigned two different teachers who teach the same subject violates the original rule, although each component's local key constraints can still hold.
- Whether this trade-off is acceptable depends on how the application enforces that remaining rule.


**Original lecture visual — PDF page 67**

[![BCNF example](../assets/slide-images/Lecture_2-page-067.png)](../assets/slide-images/Lecture_2-page-067.png)

The dependency teacher → subject violates BCNF even though all attributes belong to candidate keys.

Source: [Lecture_2.pdf, PDF page 67](../Lectures/Lecture_2.pdf#page=67).


## 16. Lossless joins and a minimal FD set (pages 53, 69-70; supplement)

- For a binary decomposition R into R1 and R2, a standard lossless-join test is whether the shared attributes functionally determine **all of R1 or all of R2**, under the original FDs.
- In Inspection/Staff, the shared staffNo determines Staff. In the BCNF example, teacher determines TaughtSubject. Both pass this test.
- A join on arbitrary columns is not automatically lossless merely because the resulting tables contain foreign keys.
- An FD set's implied dependencies can be numerous. Work with an equivalent simplified set.
- **Minimal-cover method:** split RHS sets into individual attributes; remove extraneous LHS attributes when the FD still follows; remove an FD only when it follows from the others.
- Example: `{A → B, B → C, A → C}` can drop A → C because it is implied by the first two.
- Example: AB → C can simplify to A → C only if the other dependencies justify that simplification. Do not shorten a determinant by guessing.
- These closure/minimal-cover methods supplement the lecture's page-70 advice.


### Practice C — Normalize and justify

`OrderLine(orderID, itemID, orderDate, customerID, customerName, itemName, quantity)` has the only candidate key `(orderID, itemID)` and these FDs:

```text
orderID → orderDate, customerID
customerID → customerName
itemID → itemName
(orderID, itemID) → quantity
```

1. Identify the partial dependencies and the transitive chain.
2. Give a 3NF decomposition with PKs/FKs.
3. Explain how the original rows are reconstructed.


In [ ]:
answer_c = ""
print(answer_c if answer_c.strip() else "Write your normalization reasoning in answer_c.")


<details><summary>Worked solution</summary>

- Partial dependencies: orderID determines orderDate/customerID; itemID determines itemName. customerName also depends on orderID through customerID.
- Transitive chain: orderID → customerID → customerName.

```text
Customer(customerID PK, customerName)
OrderHeader(orderID PK, orderDate, customerID FK → Customer)
Item(itemID PK, itemName)
OrderLine(orderID FK → OrderHeader, itemID FK → Item,
          quantity, PK(orderID, itemID))
```

Join OrderLine with OrderHeader on orderID, Item on itemID, and Customer on customerID. Each referenced identifier determines its own row; those joins do not invent alternative matching names/dates under these assumptions. Each original FD can be checked within one component. All components are in BCNF under the stated FDs.

</details>


## 17. Runnable SQL design practice (supplementary)

- Run setup once in a **Python 3 Jupyter kernel**. The existing `sql_practice.py` helper creates disposable **in-memory SQLite** tables; no MySQL server or extra SQL plugin is required.
- `%%sql` runs one statement per cell. Write your own answer in each comments-only starter cell, then run its check.
- Solutions are folded below the exercises. Checks test the sample and selected structural properties, not all possible business rules.
- SQLite practice uses portable key/FK ideas; dialect-specific schema alterations may differ from MySQL. `lab.reset()` restores the original sample and removes your extra practice table.


In [ ]:
from pathlib import Path
import sys

practice_folder = next((p for p in [Path.cwd(), Path.cwd().parent,
    Path.cwd() / "COMP0178", Path("/Users/peidi/Desktop/COMP0178")]
    if (p / "sql_practice.py").exists()), None)
if practice_folder is None:
    raise FileNotFoundError("Keep sql_practice.py in the COMP0178 folder.")
sys.path.insert(0, str(practice_folder))
from sql_practice import Practice
lab = Practice()
lab.register(get_ipython())


### Practice D — Translate a multi-valued attribute

The existing Branch table has branchNo as its PK. Create **BranchPhone** with branchNo and telNo, both NOT NULL, a composite PK, and a FK to Branch. Use `CREATE TABLE IF NOT EXISTS` so the cell can run again. Do not store comma-separated phone numbers.


In [ ]:
%%sql
-- Write your CREATE TABLE IF NOT EXISTS BranchPhone statement here.


In [ ]:
# Run after your CREATE TABLE cell.
info = lab.connection.execute("PRAGMA table_info(BranchPhone)").fetchall()
fks = lab.connection.execute("PRAGMA foreign_key_list(BranchPhone)").fetchall()
if not info:
    print("Create BranchPhone in the SQL answer cell first.")
else:
    pk = [row[1] for row in sorted(info, key=lambda row: row[5]) if row[5]]
    assert pk == ["branchNo", "telNo"], "Use the composite primary key (branchNo, telNo)."
    assert all(row[3] == 1 for row in info if row[1] in ["branchNo", "telNo"]), "Both columns must be NOT NULL."
    assert any(row[2:5] == ("Branch", "branchNo", "branchNo") for row in fks), "Add the FK to Branch(branchNo)."
    print("The table has the expected key, NOT NULL columns and foreign key.")


<details><summary>SQL solution and explanation</summary>

```sql
CREATE TABLE IF NOT EXISTS BranchPhone (
    branchNo TEXT NOT NULL,
    telNo TEXT NOT NULL,
    PRIMARY KEY (branchNo, telNo),
    FOREIGN KEY (branchNo) REFERENCES Branch(branchNo)
);
```

The same branch can have several phone numbers, but the same pair cannot be repeated. This definition does not enforce a maximum of three numbers or a minimum of one; those need further logic.

</details>


### Practice E — Use the relationship table

Return client first name, property street and viewing date for every viewing. Use **Client, Viewing and Property**, in that column order. Order by clientNo, propertyNo and viewDate.


In [ ]:
%%sql
-- Write your SELECT using the three tables here.


In [ ]:
lab.check([
    ("Mia", "10 Dawn Rd", "2024-05-01"),
    ("Mia", "20 Park Rd", "2024-05-02"),
    ("Noah", "20 Park Rd", "2024-05-03"),
    ("Noah", "40 Park St", "2024-05-04"),
], ordered=True)


<details><summary>SQL solution and explanation</summary>

```sql
SELECT c.fName, p.street, v.viewDate
FROM Client AS c
JOIN Viewing AS v ON v.clientNo = c.clientNo
JOIN Property AS p ON p.propertyNo = v.propertyNo
ORDER BY v.clientNo, v.propertyNo, v.viewDate;
```

Viewing records the many-to-many association and its date/comment. The other tables store the client and property facts. The four expected rows are printed by the check; no DISTINCT is needed to hide distinct viewing events.

</details>


## 18. Exam connections — actual papers, not predictions

| Paper | Exact question | Connection to this lecture |
| --- | --- | --- |
| [2023](../Exampapers/COMP0178A7PE.pdf#page=2) | Q1, 15 marks | Decide whether a supplied school schema is in 3NF; identify violations and repairs |
| [2024](../Exampapers/COMP0178A7PF.pdf#page=2) | Q1, 15 marks | Translate the given school ER diagram to a schema and explain the translation in 2–3 sentences |
| [2025](../Exampapers/COMP0178A7PG.pdf#page=2) | Q1, 20 marks | Draw an ER diagram from restaurant-review requirements, with relationship multiplicities |

**For normalization:** state the business-rule FDs and candidate keys, name the violation, then show the repaired schemas and key/FK links. Do not just assert “3NF.”

**For translation:** read role names and both endpoint ranges before choosing FK placement. Include relationship attributes and identify composite keys.

**For requirements → ERD:** turn nouns into candidate entity types, verbs into relationships, and “each / optional / many” statements into multiplicities. State assumptions where the specification does not fix them.


**Original exam visual — PDF page 2**

[![2024 Q1 ER diagram](../assets/slide-images/COMP0178A7PF-page-002.png)](../assets/slide-images/COMP0178A7PF-page-002.png)

Use the exact diagram when translating the 2024 Q1 school system; read the roles and multiplicities before choosing foreign keys.

Source: [COMP0178A7PF.pdf, PDF page 2](../Exampapers/COMP0178A7PF.pdf#page=2).


### Exam practice: 2024 Q1 — translate the exact diagram

Before opening the answer, write the four entity tables, then translate Contains, Held in and Teaches. Where should dayOfWeek and time go?

<details><summary>One translation with a stated scheduling assumption</summary>

```text
Classroom(roomID PK, location, equipment)
Teacher(staffID PK, firstName, lastName)
Class(classCode PK, title, description, staffID NOT NULL FK → Teacher)
Student(studentID PK, firstName, lastName)
Enrollment(classCode FK → Class, studentID FK → Student,
           PK(classCode, studentID))
HeldIn(roomID FK → Classroom, classCode FK → Class, dayOfWeek, time,
       PK(roomID, classCode))
```

This assumes one HeldIn relationship instance per classroom/class pair, as in the binary ER model. If several scheduled meetings for the same pair must be stored, use an event ID or include sufficient scheduling attributes in the key; state that additional requirement. It also assumes equipment is a single-valued attribute: if it is a list of equipment items, model it separately.

**Three-sentence explanation:** Each entity becomes a table with its shown primary key. Contains and Held in are many-to-many relationships, so they become junction tables; dayOfWeek and time describe Held in and are stored there. Teaches is one-to-many with exactly one teacher per class, so Class holds a NOT NULL staffID foreign key, without UNIQUE because a teacher may teach many classes.

</details>


### Exam practice: 2023 Q1 — inspect the actual schema

```text
Teachers(teacherID, fName, lName, email)
Students(studentID, fName, lName, year, email)
Module(moduleID, moduleCode, moduleName, subject, desc)
Class(classID, moduleID, term, time, listOfStudents)
Teaching(teacherID, classID, moduleCode, term, classroom)
```

The paper states that each class has exactly one teacher, a fixed daily time and a fixed classroom. Modules may have several classes; a student takes only one class per module. Read the [full question](../Exampapers/COMP0178A7PE.pdf#page=2) before answering.

- Identify the obvious 1NF problem.
- Write the dependencies involving classID and moduleID.
- Propose one repaired design and state any assumptions about module identity and student enrollment.


In [ ]:
answer_exam_2023 = ""
print(answer_exam_2023 if answer_exam_2023.strip() else "Write the violations, FDs and repaired schema here.")


<details><summary>One justified repair — not an official marking scheme</summary>

- listOfStudents is a repeating group; use a separate Enrollment relation.
- classID → moduleID, term, time, teacherID, classroom follows from the stated class rules.
- moduleID → moduleCode, moduleName, subject, desc. If moduleCode is unique, it is an alternate candidate key; do not assume that without a rule.
- In Teaching, `(teacherID, classID)` is not minimal because classID already determines teacherID. Class facts should not be duplicated in a relationship table. Under that key, classID → moduleCode through moduleID is a transitive dependency.

A possible repair:

```text
Teachers(teacherID PK, fName, lName, email)
Students(studentID PK, fName, lName, year, email)
Module(moduleID PK, moduleCode, moduleName, subject, desc)
Class(classID PK, moduleID FK → Module, term, time,
      teacherID FK → Teachers, classroom)
Enrollment(studentID FK → Students, classID FK → Class,
           PK(studentID, classID))
```

This assumes IDs identify the stated objects and no further FDs among their descriptive attributes. Recover moduleCode via the Module FK rather than copying it into Teaching. If every class must have a teacher, teacherID is NOT NULL. The rule “one class per student per module” is **not enforced by Enrollment's pair PK alone**; it needs an additional cross-table check or a carefully designed alternative enrollment schema. State how it is enforced rather than claiming the FK solves it. If module/term uniquely determines a term elsewhere, revisit where term belongs.

</details>


### Exam practice: 2025 Q1 — restaurant-review app

Draw User, Restaurant, Review and RestaurantList, plus an association for list membership. Include rating/text, list identity/title and public/private visibility. Use the [original requirements](../Exampapers/COMP0178A7PG.pdf#page=2).

<details><summary>One defensible design with explicit assumptions</summary>

- User **1 — 0..*** Review; Restaurant **1 — 0..*** Review. Each Review refers to exactly one user and one restaurant.
- User **1 — 0..*** RestaurantList. Each list has exactly one owner.
- RestaurantList **0..* — 0..*** Restaurant via ListEntry. Each ListEntry has exactly one list and one restaurant.
- Suggested attributes: User(userID), Restaurant(restaurantID, name, address, latitude, longitude, price, cuisine), Review(reviewID, rating, text), RestaurantList(listID, title, visibility).
- Assumptions: empty lists and restaurants without reviews are allowed; a restaurant occurs at most once within a given list. If new lists must contain an entry, change the relevant minimum from 0 to 1.
- The paper does not fix how many reviews a user may write for the same restaurant. Allow repeats with reviewID, or explicitly impose UNIQUE(userID, restaurantID) if assuming one review per pair.
- A list title is not its unique identifier; use listID as specified. If cuisine/features can have multiple values, model them with separate attribute/association tables as needed.

</details>


## 19. Revision checklist and answers (page 71 recap)

Try answering before opening the solutions:

1. What is the difference between conceptual and logical design?
2. How do you read `0..*` near Property in a Staff–Property relationship?
3. What distinguishes a composite key from a composite attribute?
4. Where does the FK go for 1:many? What extra constraint makes it 1:1?
5. Why does a many:many relationship need a separate table?
6. Why can sample data refute, but not prove, a proposed business-rule FD?
7. What violates 2NF? What violates 3NF?
8. Why is the lecture's Teaching relation in 3NF but not BCNF?
9. What is the difference between losslessness and dependency preservation?
10. What does an inheritance arrow mean, and what do `and`/`or` mean?

<details><summary>Answers</summary>

1. Conceptual design models meanings and rules; logical design specifies tables/columns/keys for the chosen data model.
2. Fix one Staff: zero or more properties may be linked to it.
3. A composite key is a multi-attribute identifier; a composite attribute has component parts, such as an address.
4. FK goes on the many side. UNIQUE limits it to one child per referenced parent; NOT NULL enforces the child's mandatory participation.
5. Neither participant alone can store all associations without repeating groups or duplicate entity facts; the junction holds the pairs/events and relationship attributes.
6. One conflicting pair refutes an FD in the sample; absence of conflicts does not establish a rule for future legal states.
7. 2NF forbids partial dependency of non-prime attributes on candidate keys. 3NF requires every nontrivial X → A to have a superkey X or a prime A.
8. teacher → subject has a prime RHS, satisfying 3NF, but teacher is not a superkey, violating BCNF.
9. Losslessness preserves exactly the original rows under reconstruction; dependency preservation allows enforcing the original FDs locally on components.
10. Arrow points from subtype to supertype. `and` allows overlapping subtype memberships; `or` makes them disjoint. Total/partial participation is a separate condition.

</details>
